In [ ]:
import os
import sys
import difflib
from datetime import datetime
import re
import collections # Using deque for recursive search

# Ensure the current working directory is in the Python path for imports
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

#from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsList, PdsComment, PdsOperatorCondition, PdsNode
from pds_parser import (
    PdsParser, PdsBlock, PdsKeyValuePair, PdsList, 
    PdsComment, PdsBlankLine, PdsOperatorCondition, PdsNode
)
from pds_differ import PdsDiffer, PdsChange

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print("-" * 80)

# --- Paths (UNCHANGED) ---
SIEGE_EVENTS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\events\siege_events.txt"
SIEGE_EVENTS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\events\siege_events.txt"
SIEGE_EVENTS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\events\siege_events.txt"
INNOVATIONS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\common\culture\innovations\00_tribal_innovations.txt"

# --- File I/O Helpers (UNCHANGED) ---
def get_file_content(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f: return f.read()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='utf-8') as f: return f.read()
        except Exception as e_inner: print(f"ERROR reading {filepath} (fallback): {e_inner}", file=sys.stderr); return None
    except FileNotFoundError: print(f"WARNING: File not found: {filepath}", file=sys.stderr); return None
    except Exception as e: print(f"ERROR reading {filepath}: {e}", file=sys.stderr); return None

def write_to_file(filepath, content):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    try:
        with open(filepath, 'w', encoding='utf-8-sig') as f: f.write(content); return True
    except Exception as e: print(f"ERROR writing to {filepath}: {e}", file=sys.stderr); return False

# --- Tree Navigation Helpers ---

def _parse_indexed_identifier_for_nav(identifier_full):
    """Helper to parse 'key___index' or just 'key' for navigation utilities."""
    if isinstance(identifier_full, str) and "___" in identifier_full:
        parts = identifier_full.split("___", 1)
        base_key = parts[0]
        try:
            index = int(parts[1])
            return base_key, index, True # is_indexed
        except ValueError: # "___" was part of the key itself
            return identifier_full, 0, False # Treat as non-indexed key
    # Ensure it handles non-string inputs gracefully if they can occur, though path segments should be strings.
    return str(identifier_full) if identifier_full is not None else None, 0, False 

def _get_key_from_path_segment(segment_str: str) -> str:
    """Extracts the base key from a 'key___index' or 'key___INSERTED...' segment."""
    if "___" in segment_str:
        return segment_str.split("___", 1)[0]
    return segment_str

def find_container_by_key_path(root_nodes_list_or_block, key_name_path: list[str]):
    """
    Finds a container PdsBlock by a sequence of key names.
    Ignores indices, takes the first match for each key.
    Used to find parent blocks for ADD operations more robustly.
    key_name_path: A list of strings, e.g., ["innovation_longboats", "character_modifier"]
    """
    current_container_items = root_nodes_list_or_block
    target_block = None

    for i, key_name in enumerate(key_name_path):
        found_next_block = None
        
        items_to_search = []
        if isinstance(current_container_items, list): # Root list or PdsBlock.children
            items_to_search = current_container_items
        elif isinstance(current_container_items, PdsBlock): # Should have been .children already
            items_to_search = current_container_items.children
        else: # Should not happen if path is correct
            print(f"  DEBUG (find_container_by_key_path): Expected list or PdsBlock, got {type(current_container_items)} for key '{key_name}'")
            return None

        for item in items_to_search:
            if isinstance(item, PdsBlock) and item.key == key_name:
                found_next_block = item
                break
            # Also check PdsKeyValuePair/PdsList/PdsOperatorCondition if their key matches
            # and we are not at the last segment (expecting a PdsBlock container)
            elif hasattr(item, 'key') and item.key == key_name and i < len(key_name_path) -1 :
                 if isinstance(item.value, PdsBlock): # KVP whose value is a block
                     found_next_block = item.value
                     break
            # Add more complex cases if a KVP can be a parent for non-block items etc.
            
        if found_next_block is None:
            #print(f"  DEBUG (find_container_by_key_path): Key '{key_name}' not found or not a PdsBlock in path.")
            return None # Key not found or not a block
        
        target_block = found_next_block
        current_container_items = target_block.children # Next search is within this block's children

    return target_block # This is the final PdsBlock in the key_name_path


def find_node_by_old_path_in_sim_tree(sim_root_nodes_list, old_key_path_list: list[str]):
    """
    Finds a specific node instance in the sim_tree using an Old-centric path (key___index).
    This is more brittle due to indices but needed if structural search fails or for context.
    """
    current_level_nodes = sim_root_nodes_list
    target_node = None

    for segment_idx, segment_full_str in enumerate(old_key_path_list):
        # Handle "___INSERTED..." segments: these refer to nodes that weren't in Old,
        # so they cannot be found by an Old-centric path lookup directly.
        # The parent should have been found by segments *before* this one.
        if "___INSERTED_at_" in segment_full_str and segment_idx == len(old_key_path_list) -1 :
             #print(f"  DEBUG (find_node_by_old_path): Path points to an INSERTED node ('{segment_full_str}'), cannot find by Old path. Parent should be used.")
             return None # Cannot find an "inserted" node by an old path logic that expects existing nodes

        # USE THE CORRECT HELPER: _parse_indexed_identifier_for_nav
        segment_key_part, target_occurrence_idx, is_indexed = _parse_indexed_identifier_for_nav(segment_full_str)

        current_key_match_count = 0
        found_in_level = False
        
        search_list = []
        if isinstance(current_level_nodes, list):
            search_list = current_level_nodes
        elif isinstance(current_level_nodes, PdsBlock):
            search_list = current_level_nodes.children
        else: # Cannot search children of non-list/non-block
            #print(f"  DEBUG (find_node_by_old_path): Cannot search for children in type {type(current_level_nodes)}")
            return None


        for node_in_sim in search_list:
            node_in_sim_key_str = None
            if hasattr(node_in_sim, 'key') and node_in_sim.key is not None:
                # Ensure consistency with how PdsDiffer._get_node_key_for_path might format keys
                key_str = str(node_in_sim.key)
                node_in_sim_key_str = key_str.replace("___", "__^__").replace(".", "^")
            elif isinstance(node_in_sim, PdsComment): # PdsComment is imported
                node_in_sim_key_str = f"__COMMENT_{hash(node_in_sim.comment_text[:20])}"
            elif isinstance(node_in_sim, PdsBlankLine): # PdsBlankLine is now imported
                node_in_sim_key_str = f"__BLANK_LINE_{node_in_sim.line_number}"
            else: # Fallback for other node types without a 'key'
                node_in_sim_key_str = f"__{node_in_sim.__class__.__name__.upper()}_L{node_in_sim.line_number}"

            if node_in_sim_key_str == segment_key_part:
                if not is_indexed or current_key_match_count == target_occurrence_idx:
                    target_node = node_in_sim
                    found_in_level = True
                    break
                current_key_match_count += 1
        
        if not found_in_level:
            #print(f"  DEBUG (find_node_by_old_path): Segment '{segment_full_str}' (key: '{segment_key_part}') not found in current level of sim_tree.")
            return None 
        
        if segment_idx < len(old_key_path_list) - 1: 
            if isinstance(target_node, PdsBlock):
                current_level_nodes = target_node # Children will be searched in next iteration
            else:
                #print(f"  DEBUG (find_node_by_old_path): Path expects children from non-PdsBlock node for segment '{segment_full_str}'. Node is {type(target_node)}.")
                return None 
    
    return target_node


def find_copied_node_recursive(sim_tree_items: list, original_node_from_new_ast: PdsNode):
    """
    Recursively searches a list of items (sim_tree_items, can be root list or PdsBlock.children)
    for a node that is structurally equal to original_node_from_new_ast.
    Returns the (copied_node_in_sim_tree, its_parent_in_sim_tree) or (None, None).
    Parent can be the list itself if node is top-level, or a PdsBlock.
    """
    if original_node_from_new_ast is None: return None, None

    queue = collections.deque()
    # Queue items are (node_to_check, parent_of_node_to_check)
    # For root items, parent is the root list itself.
    for item in sim_tree_items:
        queue.append((item, sim_tree_items))

    while queue:
        current_node, current_parent = queue.popleft()

        if current_node == original_node_from_new_ast: # Uses PdsNode.__eq__ (deep structural)
            return current_node, current_parent

        if isinstance(current_node, PdsBlock):
            for child in current_node.children:
                queue.append((child, current_node)) # Parent of child is current_node (PdsBlock)
        elif isinstance(current_node, (PdsKeyValuePair, PdsOperatorCondition)):
            if isinstance(current_node.value, PdsBlock): # KVP/OpCond whose value is a block
                 for child_of_value_block in current_node.value.children:
                      queue.append((child_of_value_block, current_node.value))


    return None, None


def find_node_and_parent_in_sim_tree(sim_root_list, chg_obj: PdsChange):
    """
    PRIMARY FUNCTION to locate target node and its parent in the simulated tree for applying a change.
    Returns: (target_node_in_sim, parent_in_sim)
    """
    target_node_in_sim = None
    parent_in_sim = None

    if chg_obj.type == 'MOD_ADDED':
        # For MOD_ADDED, the target node doesn't exist yet. We need to find the parent container.
        # Use the key names from context_parent_path.
        parent_key_names = [_get_key_from_path_segment(s) for s in chg_obj.context_parent_path]
        parent_in_sim = find_container_by_key_path(sim_root_list, parent_key_names)
        target_node_in_sim = None # It's an addition
        if parent_in_sim is None:
             print(f"  SIM MERGE FAIL (MOD_ADDED): Parent container for path '{'.'.join(parent_key_names)}' not found in sim_tree.")
        return target_node_in_sim, parent_in_sim

    # For other change types, try to find the node.
    if chg_obj.new_node is not None:
        # Strategy 1 (Primary): Find copy of original New node in sim_tree by structural equality.
        target_node_in_sim, parent_candidate = find_copied_node_recursive(sim_root_list, chg_obj.new_node)
        if target_node_in_sim:
            parent_in_sim = parent_candidate # find_copied_node_recursive now returns parent
            if parent_in_sim is None: # Should not happen if target_node_in_sim found
                 print(f"  SIM MERGE ERROR: Found target node by copy, but parent is None. Path: {'.'.join(chg_obj.key_path)}")
                 return None, None
            # Verification (optional but good): check if parent_in_sim indeed contains target_node_in_sim
            if isinstance(parent_in_sim, list) and target_node_in_sim not in parent_in_sim:
                print(f"  SIM MERGE ERROR: Found target by copy, parent (list) by upward search, but target not in parent. Path: {'.'.join(chg_obj.key_path)}")
                return None, None
            elif isinstance(parent_in_sim, PdsBlock) and target_node_in_sim not in parent_in_sim.children:
                print(f"  SIM MERGE ERROR: Found target by copy, parent (block) by upward search, but target not in parent's children. Path: {'.'.join(chg_obj.key_path)}")
                return None, None

        else: # Structural search failed
            print(f"  SIM MERGE DEBUG: Target node for new_node (key: {getattr(chg_obj.new_node, 'key', 'N/A')}) not found by structural search. Path: {'.'.join(chg_obj.key_path)}.")
            # Fallback for MOD_MODIFIED, MOD_DELETED etc. if new_node was present but not found by copy:
            # Try finding the node by its Old path in sim_tree. This is less reliable.
            # And then try finding its parent using the Old context path.
            print(f"    Attempting fallback: find node by Old path in sim_tree.")
            target_node_in_sim = find_node_by_old_path_in_sim_tree(sim_root_list, chg_obj.key_path)
            if target_node_in_sim:
                 # Now find its parent. This is tricky.
                 # Let's try finding parent by Old context path, then verify.
                 old_parent_path = chg_obj.context_parent_path
                 if not old_parent_path: # Node is root level
                     parent_in_sim = sim_root_list
                 else:
                     parent_in_sim = find_node_by_old_path_in_sim_tree(sim_root_list, old_parent_path)
                 
                 # Verification
                 if parent_in_sim is None:
                     print(f"    Fallback FAIL: Target node found by Old path, but its parent (via Old context path) not found. Path: {'.'.join(chg_obj.key_path)}")
                     return None, None
                 if isinstance(parent_in_sim, list) and target_node_in_sim not in parent_in_sim:
                     print(f"    Fallback FAIL: Target found by Old path, parent by Old path, but target not in parent list. Path: {'.'.join(chg_obj.key_path)}")
                     return None, None
                 if isinstance(parent_in_sim, PdsBlock) and target_node_in_sim not in parent_in_sim.children:
                     print(f"    Fallback FAIL: Target found by Old path, parent block by Old path, but target not in parent children. Path: {'.'.join(chg_obj.key_path)}")
                     return None, None
            else:
                 print(f"    Fallback FAIL: Target node not found by Old path either. Path: {'.'.join(chg_obj.key_path)}")
                 return None, None # Both structural and path search failed

    elif chg_obj.old_node is not None: # e.g. MOD_DELETED or VANILLA_DELETED where new_node is None but old_node existed
        # We need to find where this old_node *would have been* in the sim_tree to delete it or comment parent.
        # The target_node_in_sim will be based on the Old path.
        target_node_in_sim = find_node_by_old_path_in_sim_tree(sim_root_list, chg_obj.key_path)
        # Parent finding logic for this target_node_in_sim (if found)
        if target_node_in_sim :
            old_parent_path = chg_obj.context_parent_path
            if not old_parent_path: parent_in_sim = sim_root_list
            else: parent_in_sim = find_node_by_old_path_in_sim_tree(sim_root_list, old_parent_path)
            # Add verification as above if parent_in_sim is found
            if parent_in_sim is None : target_node_in_sim = None # Cant operate if parent not found
            elif isinstance(parent_in_sim, list) and target_node_in_sim not in parent_in_sim: target_node_in_sim = None
            elif isinstance(parent_in_sim, PdsBlock) and target_node_in_sim not in parent_in_sim.children: target_node_in_sim = None

        elif not target_node_in_sim : # If target itself not found by Old path (already gone)
            # Still try to find parent for commenting using Old context path
            old_parent_path = chg_obj.context_parent_path
            if not old_parent_path: parent_in_sim = sim_root_list
            else: parent_in_sim = find_node_by_old_path_in_sim_tree(sim_root_list, old_parent_path)
            # target_node_in_sim remains None

    else: # Should not happen: no new_node and no old_node for a change that needs location
        print(f"  SIM MERGE FAIL: No new_node or old_node to locate item for change type {chg_obj.type}. Path: {'.'.join(chg_obj.key_path)}")
        return None, None

    if parent_in_sim is not None and not isinstance(parent_in_sim, (list, PdsBlock)):
        print(f"  SIM MERGE ERROR: Parent found is invalid type {type(parent_in_sim)}. Path: {'.'.join(chg_obj.key_path)}")
        return None, None
        
    return target_node_in_sim, parent_in_sim


def normalize_for_comparison(text): # UNCHANGED
    if not text: return ""
    normalized = re.sub(r'\n(\s*\n)+', '\n', text) # Collapse multiple blank lines
    # Consider also removing leading/trailing whitespace from each line if needed
    return normalized.strip()


def run_and_print_diff(test_name, old_filepath, mod_filepath, new_filepath):
    print(f"\n{'='*20} RUNNING TEST: {test_name} {'='*20}")
    # ... (file I/O and parsing setup - UNCHANGED from your version, ensure it uses the new parser) ...
    test_output_dir = os.path.join(os.getcwd(), "test_output", test_name.replace(" ", "_").replace("(", "").replace(")", ""))
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir}")

    old_content_raw = get_file_content(old_filepath)
    mod_content_raw = get_file_content(mod_filepath)
    new_content_raw = get_file_content(new_filepath)
    # ... (save raw files - UNCHANGED) ...
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RAW.txt")), old_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RAW.txt")), mod_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RAW.txt")), new_content_raw or "")
    print(f"  Raw files saved.")

    parser = PdsParser() # Use the new parser
    print(f"  Parsing Old file: {os.path.basename(old_filepath)}")
    old_nodes = parser.parse_file(old_filepath)
    print(f"  Parsing Mod file: {os.path.basename(mod_filepath)}")
    mod_nodes = parser.parse_file(mod_filepath)
    print(f"  Parsing New file: {os.path.basename(new_filepath)}")
    new_nodes = parser.parse_file(new_filepath)

    if old_nodes is None: old_nodes = []
    if mod_nodes is None: mod_nodes = []
    if new_nodes is None: new_nodes = []

    if not (old_nodes or mod_nodes or new_nodes): # Simpler check if all are empty
        if not (old_content_raw or mod_content_raw or new_content_raw):
             print(f"SKIPPING: No content or parseable ASTs for test '{test_name}'.")
             return
        else:
             print(f"WARNING: Some raw content exists but ASTs are all empty for '{test_name}'. Check parser errors.")


    reconstructed_old = PdsParser._nodes_to_string(old_nodes)
    reconstructed_mod = PdsParser._nodes_to_string(mod_nodes)
    reconstructed_new = PdsParser._nodes_to_string(new_nodes)
    # ... (save reconstructed files - UNCHANGED) ...
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RECONSTRUCTED.txt")), reconstructed_old)
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RECONSTRUCTED.txt")), reconstructed_mod)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RECONSTRUCTED.txt")), reconstructed_new)
    print(f"  Reconstructed files saved.")
    
    normalized_old_raw = normalize_for_comparison(old_content_raw or "")
    normalized_mod_raw = normalize_for_comparison(mod_content_raw or "")
    normalized_new_raw = normalize_for_comparison(new_content_raw or "")

    if reconstructed_old.strip() != normalized_old_raw: print(f"\nWARNING: Reconstruction mismatch for OLD file: {os.path.basename(old_filepath)}.")
    if reconstructed_mod.strip() != normalized_mod_raw: print(f"\nWARNING: Reconstruction mismatch for MOD file: {os.path.basename(mod_filepath)}.")
    if reconstructed_new.strip() != normalized_new_raw: print(f"\nWARNING: Reconstruction mismatch for NEW file: {os.path.basename(new_filepath)}.")

    differ = PdsDiffer() # Use the new differ
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes)

    print(f"\n--- DETECTED CHANGES for '{test_name}' ({len(changes)} changes) ---")
    if not changes: print("    No significant changes detected by differ.")
    for change in changes: print(change)

    # --- Simulation ---
    simulated_merged_nodes_root_list = [node.copy() for node in new_nodes] 

    def _simulate_apply_change(sim_tree_root_list, chg_obj: PdsChange):
        print(f"\n  Attempting to apply change: {chg_obj.type} at OldPath {'->'.join(chg_obj.key_path)}")
        sim_comment_text = f"SimMerge:{chg_obj.type}"

        def _add_comment_to_node(node, comment_str):
            if node is None: return
            # Simplified: always append. Could be smarter to avoid duplicates.
            if hasattr(node, 'comment_text_on_line'):
                current_comment = node.comment_text_on_line
                node.comment_text_on_line = f"{current_comment} {comment_str}" if current_comment else comment_str
            elif isinstance(node, PdsBlock): # Add comment as first child if block has no line comment
                comment_node = PdsComment(comment_str)
                node.children.insert(0, comment_node)


        target_node_in_sim, parent_in_sim = find_node_and_parent_in_sim_tree(
            sim_tree_root_list, chg_obj
        )
        
        # Path for debug output (Old-centric)
        debug_path_str = '->'.join(chg_obj.key_path) if chg_obj.key_path else "ROOT"

        if chg_obj.type != 'MOD_ADDED' and target_node_in_sim is None :
             # For most changes, if target_node_in_sim is None, we likely can't proceed.
             # MOD_ADDED is an exception as target_node_in_sim will be None.
             # Deletions might also have target_node_in_sim as None if already absent.
             if not (chg_obj.type.endswith("_DELETED") or chg_obj.type.endswith("_ALSO_DELETED")):
                print(f"  SIM MERGE FAIL ({chg_obj.type}): Target node not found in sim_tree for path '{debug_path_str}'. Skipping.")
                return

        if parent_in_sim is None and chg_obj.type not in ('VANILLA_DELETED', 'MOD_DELETED_VANILLA_ALSO_DELETED'): # Parent is crucial for most ops
             # For some deletions, parent might be commented even if node is gone.
             # If target_node_in_sim is also None (already gone), and parent is None, that's a deeper issue.
             if not (target_node_in_sim is None and chg_obj.type.endswith("_DELETED")):
                 print(f"  SIM MERGE FAIL ({chg_obj.type}): Parent container not found in sim_tree for path '{debug_path_str}'. Skipping.")
                 return

        print(f"    Path: {debug_path_str}, Parent Found: {parent_in_sim is not None} (Type: {type(parent_in_sim).__name__}), TargetInSim Found: {target_node_in_sim is not None}")

        # --- Apply Mod-specific changes ---
        if chg_obj.type == 'MOD_MODIFIED':
            if target_node_in_sim and parent_in_sim and chg_obj.mod_node:
                mod_node_copy = chg_obj.mod_node.copy()
                _add_comment_to_node(mod_node_copy, sim_comment_text)
                if isinstance(parent_in_sim, list): # Root node
                    try:
                        idx = parent_in_sim.index(target_node_in_sim)
                        mod_node_copy.indent_level = 0
                        parent_in_sim[idx] = mod_node_copy
                        print(f"      Applied MOD_MODIFIED: Replaced root node.")
                    except ValueError: print(f"    SIM MERGE ERROR (MOD_MOD Root): Target not found in parent list for replacement.")
                elif isinstance(parent_in_sim, PdsBlock):
                    if parent_in_sim.replace_child(target_node_in_sim, mod_node_copy): # Pass instance
                        print(f"      Applied MOD_MODIFIED: Replaced child in block '{parent_in_sim.key}'.")
                    else: print(f"    SIM MERGE ERROR (MOD_MOD Block): PdsBlock.replace_child failed.")
            else: print(f"    SIM MERGE FAIL (MOD_MODIFIED): Target, parent, or mod_node missing.")

        elif chg_obj.type == 'MOD_ADDED':
            if parent_in_sim and chg_obj.mod_node:
                mod_node_copy = chg_obj.mod_node.copy()
                _add_comment_to_node(mod_node_copy, sim_comment_text)
                if isinstance(parent_in_sim, list): # Adding to root
                    mod_node_copy.indent_level = 0
                    parent_in_sim.append(mod_node_copy)
                    print(f"      Applied MOD_ADDED: Appended root node.")
                elif isinstance(parent_in_sim, PdsBlock):
                    parent_in_sim.add_child_at_appropriate_location(mod_node_copy)
                    print(f"      Applied MOD_ADDED: Added child to block '{parent_in_sim.key}'.")
                else: print(f"    SIM MERGE ERROR (MOD_ADDED): Parent is not a list or PdsBlock.")
            else: print(f"    SIM MERGE FAIL (MOD_ADDED): Parent or mod_node missing.")
        
        elif chg_obj.type == 'MOD_DELETED':
            if target_node_in_sim and parent_in_sim: # Node must exist in sim_tree to be deleted by Mod
                if isinstance(parent_in_sim, list):
                    try: parent_in_sim.remove(target_node_in_sim); print(f"      Applied MOD_DELETED: Removed root node.")
                    except ValueError: print(f"    SIM MERGE ERROR (MOD_DEL Root): Target not found in parent list for deletion.")
                elif isinstance(parent_in_sim, PdsBlock):
                    if parent_in_sim.remove_child(target_node_in_sim): print(f"      Applied MOD_DELETED: Removed child from block '{parent_in_sim.key}'.")
                    else: print(f"    SIM MERGE ERROR (MOD_DEL Block): PdsBlock.remove_child failed.")
            elif target_node_in_sim is None: # Already absent, that's fine for a deletion
                print(f"      Applied MOD_DELETED: Target already absent. Commenting parent if found.")
                _add_comment_to_node(parent_in_sim, f"{sim_comment_text} (target already absent)")
            else: print(f"    SIM MERGE FAIL (MOD_DELETED): Parent missing for target.")


        # --- Converged changes and Vanilla changes: already in New tree, just comment ---
        elif chg_obj.type in ('CONVERGED_MODIFICATION', 'MOD_ADDED_CONVERGED', 
                              'VANILLA_MODIFIED', 'VANILLA_ADDED'):
            if target_node_in_sim: # Should be found via structural search of new_node
                _add_comment_to_node(target_node_in_sim, sim_comment_text)
                print(f"      Applied {chg_obj.type}: Added comment to node.")
            else: print(f"    SIM MERGE WARN ({chg_obj.type}): Node not found in sim_tree to add comment (should have been from new_node).")

        elif chg_obj.type in ('VANILLA_DELETED', 'MOD_DELETED_VANILLA_ALSO_DELETED'):
            print(f"      Applied {chg_obj.type}: Node already absent from sim_tree. Commenting parent if found.")
            _add_comment_to_node(parent_in_sim, f"{sim_comment_text} (target absent)")
            
        # --- Conflicts ---
        elif chg_obj.type.startswith('CONFLICT'):
            print(f"    Handling {chg_obj.type}...")
            # CONFLICT_MODIFIED: Mod vs Vanilla. Prefer Mod.
            if chg_obj.type == 'CONFLICT_MODIFIED':
                if target_node_in_sim and parent_in_sim and chg_obj.mod_node:
                    mod_node_copy = chg_obj.mod_node.copy()
                    _add_comment_to_node(mod_node_copy, sim_comment_text + "_MOD_CHOSEN")
                    if isinstance(parent_in_sim, list):
                        try: idx = parent_in_sim.index(target_node_in_sim); parent_in_sim[idx] = mod_node_copy; print(f"        Resolved {chg_obj.type}: Replaced with Mod's version (root).")
                        except ValueError: print(f"      SIM MERGE CONFLICT ERROR: Target not in parent list.")
                    elif isinstance(parent_in_sim, PdsBlock):
                        if parent_in_sim.replace_child(target_node_in_sim, mod_node_copy): print(f"        Resolved {chg_obj.type}: Replaced with Mod's version in block '{parent_in_sim.key}'.")
                        else: print(f"      SIM MERGE CONFLICT ERROR: PdsBlock.replace_child failed.")
                else: print(f"      SIM MERGE CONFLICT FAIL: Cannot apply Mod's version for {chg_obj.type} due to missing elements.")
            
            # CONFLICT_ADDITION: Mod added, Vanilla added differently. Prefer Mod.
            elif chg_obj.type == 'CONFLICT_ADDITION':
                if parent_in_sim and chg_obj.mod_node: # Vanilla's add is already in sim_tree (target_node_in_sim)
                                                     # We need to replace Vanilla's add with Mod's add if possible, or just add Mod's.
                                                     # This depends on how CONFLICT_ADDITION is defined.
                                                     # Assuming it means they tried to add to same "slot".
                                                     # If target_node_in_sim represents Vanilla's ADDED node:
                    if target_node_in_sim: # Vanilla's added node exists
                         print(f"        Replacing Vanilla's added node with Mod's for {chg_obj.type}.")
                         # Treat as replacing Vanilla's add
                         mod_node_copy = chg_obj.mod_node.copy()
                         _add_comment_to_node(mod_node_copy, sim_comment_text + "_MOD_CHOSEN")
                         if isinstance(parent_in_sim, list):
                             try: idx = parent_in_sim.index(target_node_in_sim); parent_in_sim[idx] = mod_node_copy
                             except ValueError: parent_in_sim.append(mod_node_copy) # If Vanilla's add wasn't found, append Mod's
                         elif isinstance(parent_in_sim, PdsBlock):
                             if not parent_in_sim.replace_child(target_node_in_sim, mod_node_copy):
                                 parent_in_sim.add_child_at_appropriate_location(mod_node_copy) # Fallback to add
                         print(f"        Resolved {chg_obj.type}: Applied Mod's addition.")
                    else: # Vanilla's added node not clearly identified, just add Mod's
                        mod_node_copy = chg_obj.mod_node.copy()
                        _add_comment_to_node(mod_node_copy, sim_comment_text + "_MOD_CHOSEN_APPENDED")
                        if isinstance(parent_in_sim, list): parent_in_sim.append(mod_node_copy)
                        elif isinstance(parent_in_sim, PdsBlock): parent_in_sim.add_child_at_appropriate_location(mod_node_copy)
                        print(f"        Resolved {chg_obj.type}: Added Mod's version (Vanilla's add not distinctly replaced).")

                else: print(f"      SIM MERGE CONFLICT FAIL: Cannot apply Mod's version for {chg_obj.type} due to missing parent/mod_node.")

            # CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED: Mod deleted, Vanilla modified. Keep Vanilla's.
            elif chg_obj.type == 'CONFLICT_DELETION_MOD_DELETED_VANILLA_MODIFIED':
                if target_node_in_sim: # This is Vanilla's modified node
                    _add_comment_to_node(target_node_in_sim, sim_comment_text + "_VANILLA_MOD_KEPT")
                    print(f"        Resolved {chg_obj.type}: Kept Vanilla's modified version.")
                else: print(f"      SIM MERGE CONFLICT WARN: Vanilla's modified node not found for {chg_obj.type}.")
            
            # CONFLICT_DELETION_VANILLA_DELETED_MOD_MODIFIED: Vanilla deleted, Mod modified. Apply Mod's (as an add).
            elif chg_obj.type == 'CONFLICT_DELETION_VANILLA_DELETED_MOD_MODIFIED':
                if parent_in_sim and chg_obj.mod_node: # Parent where Old node was
                    mod_node_copy = chg_obj.mod_node.copy()
                    _add_comment_to_node(mod_node_copy, sim_comment_text + "_MOD_MOD_APPLIED")
                    if isinstance(parent_in_sim, list): parent_in_sim.append(mod_node_copy)
                    elif isinstance(parent_in_sim, PdsBlock): parent_in_sim.add_child_at_appropriate_location(mod_node_copy)
                    print(f"        Resolved {chg_obj.type}: Applied Mod's modified version (as add to parent).")
                else: print(f"      SIM MERGE CONFLICT FAIL: Cannot apply Mod's modified version for {chg_obj.type} due to missing parent/mod_node.")
            else:
                print(f"    SIM MERGE WARN: Unhandled conflict type logic: {chg_obj.type}")
        else:
            print(f"    SIM MERGE NOTE: Change type '{chg_obj.type}' not requiring direct structural action or already handled.")

    # End of _simulate_apply_change

    print(f"\n--- SIMULATING MERGE for '{test_name}' ---")
    for change_item in changes:
        _simulate_apply_change(simulated_merged_nodes_root_list, change_item)

    simulated_merged_content = PdsParser._nodes_to_string(simulated_merged_nodes_root_list)
    # ... (save simulated merged and diff vs new - UNCHANGED) ...
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_SIMULATED_MERGED.txt")), simulated_merged_content)
    print(f"\n  Simulated merged content saved.")

    print(f"\n--- DIFF: SIMULATED MERGED vs NORMALIZED NEW VANILLA RAW for '{test_name}' ---")
    diff_filename = os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_MERGED_VS_NEW_RAW.diff"))
    
    # Normalize simulated_merged_content for a fairer diff against normalized_new_raw
    normalized_simulated_merged = normalize_for_comparison(simulated_merged_content)
    
    with open(diff_filename, 'w', encoding='utf-8') as f_diff:
        diff_lines = list(difflib.unified_diff(
            normalized_new_raw.splitlines(keepends=True),
            #simulated_merged_content.strip().splitlines(keepends=True), # Use normalized version
            normalized_simulated_merged.splitlines(keepends=True),
            fromfile='NORMALIZED_NEW_VANILLA_RAW', tofile='NORMALIZED_SIMULATED_MERGED', lineterm=''
        ))
        if diff_lines:
            f_diff.writelines(diff_lines)
            print(f"  Diff (Normalized) saved to: {os.path.basename(diff_filename)}")
            # for line in diff_lines: # Optionally print to console for quick view
            #    sys.stdout.write(line)
        else:
            print("  NORMALIZED SIMULATED MERGED is identical to NORMALIZED NEW VANILLA RAW (or only cosmetic/comment changes).")
    print("-" * 80)


# --- Run Tests ---
print("Running diff tests with real CK3 files.")
# run_and_print_diff("Siege Events (real files)",
#                    SIEGE_EVENTS_OLD_VANILLA_PATH, SIEGE_EVENTS_MOD_PATH, SIEGE_EVENTS_NEW_VANILLA_PATH)
run_and_print_diff("00_tribal_innovations (real files)",
                   INNOVATIONS_OLD_VANILLA_PATH, INNOVATIONS_MOD_PATH, INNOVATIONS_NEW_VANILLA_PATH)
print("\n--- All Tests Complete ---")

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--------------------------------------------------------------------------------
Running diff tests with real CK3 files.

==================== RUNNING TEST: 00_tribal_innovations (real files) ====================
Output files for this test will be saved to: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\test_output\00_tribal_innovations_real_files
  Raw files saved.
  Parsing Old file: 00_tribal_innovations.txt
  Parsing Mod file: 00_tribal_innovations.txt
  Parsing New file: 00_tribal_innovations.txt
  Reconstructed files saved.




--- DETECTED CHANGES for '00_tribal_innovations (real files)' (41 changes) ---
PdsChange(Type='VANILLA_MODIFIED                             ', Path='innovation_bannus___0', 
          ParentCtx='ROOT_PARENT', 
          Nodes=[
            O:PdsBlock L83 I0 innovation_bannus={...},
            M:PdsBlock L83 I0 innovation_bannus={...},
     